# 07 - Orquestacion del Pipeline
## Ejecucion Secuencial de Todos los Notebooks

---

### Objetivo de este notebook

Este notebook actua como **orquestador** del pipeline ELT completo.
Ejecuta secuencialmente todos los notebooks del pipeline en el orden correcto,
manejando errores y midiendo tiempos de ejecucion.

### Por que un orquestador?

En un pipeline de produccion, es comun tener un punto de entrada unico que:
- Garantiza el **orden correcto** de ejecucion de las etapas
- Implementa **manejo de errores** centralizado (si un paso falla, se detiene todo)
- Registra **metricas de ejecucion** (duracion, estado de cada paso)
- Facilita la **automatizacion** mediante Databricks Jobs o triggers externos

### Flujo de ejecucion

| Paso | Notebook | Descripcion |
|------|----------|-------------|
| 1 | 00_configuracion | Crear esquemas y volumenes |
| 2 | 01_ingesta_raw | Extraer datos de PubChem API |
| 3 | 02_raw_a_bronce | Cargar Raw a Delta Tables (Bronce) |
| 4 | 03_exploracion_datos | Perfilamiento de datos Bronce |
| 5 | 04_validacion_calidad | Compuerta de calidad |
| 6 | 05_bronce_a_plata | Transformacion Bronce a Plata |
| 7 | 06_plata_a_oro | Modelo estrella en capa Oro |
| 8 | 08_logs_auditoria | Registro de auditoria |

### Comportamiento ante errores

Si alguno de los notebooks falla durante su ejecucion, el pipeline
se **detiene inmediatamente** y reporta el error. No se continua
con los pasos siguientes para evitar ejecutar sobre datos incompletos.

---
## 1. Importacion de Librerias y Configuracion

In [ ]:
from datetime import datetime  # Para medir tiempos de ejecucion
import glob as modulo_glob     # Para busqueda de archivos (verificacion)

In [ ]:
# Detectar catalogo activo
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]

# Obtener la ruta base de los notebooks en el workspace de Databricks
# Se usa la API de dbutils para determinar la ubicacion del notebook actual
import os
RUTA_BASE = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
)

print(f"Catalogo: {CATALOGO}")
print(f"Ruta base de notebooks: {RUTA_BASE}")
print(f"Inicio del pipeline: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 2. Definicion del Pipeline

Se define la lista ordenada de notebooks que componen el pipeline.
Cada entrada incluye el nombre del notebook y una descripcion
de su proposito para facilitar la lectura del log de ejecucion.

In [ ]:
# Lista ordenada de notebooks a ejecutar
# El orden es critico: cada notebook depende de los anteriores
PIPELINE = [
    {"notebook": "00_configuracion",      "descripcion": "Crear infraestructura (esquemas y volumenes)"},
    {"notebook": "01_ingesta_raw",         "descripcion": "Extraer datos de PubChem API a Raw"},
    {"notebook": "02_raw_a_bronce",        "descripcion": "Cargar archivos Raw a Delta Tables (Bronce)"},
    {"notebook": "03_exploracion_datos",   "descripcion": "Perfilamiento y exploracion de datos"},
    {"notebook": "04_validacion_calidad",  "descripcion": "Compuerta de calidad sobre capa Bronce"},
    {"notebook": "05_bronce_a_plata",      "descripcion": "Transformacion Bronce a Plata"},
    {"notebook": "06_plata_a_oro",         "descripcion": "Construccion del modelo estrella (Oro)"},
    {"notebook": "08_logs_auditoria",      "descripcion": "Registro de auditoria y metricas"},
]

print(f"Pipeline configurado con {len(PIPELINE)} pasos.")

---
## 3. Ejecucion Secuencial del Pipeline

Se ejecuta cada notebook en orden usando `dbutils.notebook.run()`.
Para cada paso se registra:
- Hora de inicio y fin
- Duracion en segundos
- Estado (EXITOSO o FALLIDO)

El timeout por notebook es de **15 minutos** (900 segundos).
Si un notebook excede este tiempo, se cancela automaticamente.

In [ ]:
# Timeout maximo por notebook en segundos (15 minutos)
TIMEOUT = 900

# Lista para acumular resultados de ejecucion de cada paso
resultados_ejecucion = []

# Registrar el inicio del pipeline completo
inicio_pipeline = datetime.now()

# Encabezado del log de ejecucion
print("=" * 70)
print("           EJECUCION DEL PIPELINE ELT - PUBCHEM")
print("=" * 70)
print()

# Ejecutar cada notebook en secuencia
for i, paso in enumerate(PIPELINE, 1):
    nombre = paso["notebook"]
    ruta_notebook = f"{RUTA_BASE}/{nombre}"  # Ruta completa en el workspace
    inicio_paso = datetime.now()

    # Mostrar informacion del paso actual
    print(f"[{i}/{len(PIPELINE)}] Ejecutando: {nombre}")
    print(f"         {paso['descripcion']}")
    print(f"         Inicio: {inicio_paso.strftime('%H:%M:%S')}")

    try:
        # Ejecutar el notebook hijo con dbutils.notebook.run()
        # Esta funcion bloquea hasta que el notebook finaliza o expira el timeout
        resultado = dbutils.notebook.run(ruta_notebook, TIMEOUT)

        # Calcular duracion del paso
        fin_paso = datetime.now()
        duracion = (fin_paso - inicio_paso).total_seconds()

        # Registrar resultado exitoso
        resultados_ejecucion.append({
            "paso": i,
            "notebook": nombre,
            "estado": "EXITOSO",
            "duracion_seg": round(duracion, 1)
        })
        print(f"         Estado: EXITOSO ({duracion:.1f}s)")
        print()

    except Exception as e:
        # Manejar error en la ejecucion del notebook
        fin_paso = datetime.now()
        duracion = (fin_paso - inicio_paso).total_seconds()

        # Registrar resultado fallido
        resultados_ejecucion.append({
            "paso": i,
            "notebook": nombre,
            "estado": "FALLIDO",
            "duracion_seg": round(duracion, 1)
        })

        # Mostrar error y detener el pipeline
        print(f"         Estado: FALLIDO ({duracion:.1f}s)")
        print(f"         Error: {str(e)[:200]}")
        print()
        print("PIPELINE DETENIDO: Se encontro un error en la ejecucion.")
        break  # Salir del bucle para no ejecutar pasos posteriores

# Calcular duracion total del pipeline
fin_pipeline = datetime.now()
duracion_total = (fin_pipeline - inicio_pipeline).total_seconds()

---
## 4. Resumen de Ejecucion

Se presenta un resumen tabular con el estado y duracion de cada paso
del pipeline, junto con totales globales.

In [ ]:
# Encabezado del resumen de ejecucion
print("=" * 70)
print("           RESUMEN DE EJECUCION DEL PIPELINE")
print("=" * 70)
print()

# Cabecera de la tabla de resultados
print(f"{'Paso':>4s}  {'Notebook':30s}  {'Estado':10s}  {'Duracion':>10s}")
print("-" * 60)

# Mostrar cada paso con su resultado
for resultado in resultados_ejecucion:
    print(
        f"{resultado['paso']:>4d}  "
        f"{resultado['notebook']:30s}  "
        f"{resultado['estado']:10s}  "
        f"{resultado['duracion_seg']:>8.1f}s"
    )

# Calcular totales por estado
exitosos = sum(1 for r in resultados_ejecucion if r["estado"] == "EXITOSO")
fallidos = sum(1 for r in resultados_ejecucion if r["estado"] == "FALLIDO")

# Mostrar resumen global
print("-" * 60)
print(f"Total: {exitosos} exitosos, {fallidos} fallidos")
print(f"Duracion total del pipeline: {duracion_total:.1f} segundos ({duracion_total / 60:.1f} minutos)")
print("=" * 70)

---
## 5. Resumen de Registros por Capa

Despues de la ejecucion del pipeline, se presenta un inventario de registros
en cada capa para verificar que el flujo de datos es coherente de principio a fin.

In [ ]:
# Encabezado del resumen por capa
print("=" * 70)
print("         CONTEO DE REGISTROS POR CAPA DEL PIPELINE")
print("=" * 70)
print()

# --- CAPA RAW: Contar archivos en el volumen ---
RUTA_RAW = f"/Volumes/{CATALOGO}/pubchem_raw/bioactividad"
try:
    archivos_raw = dbutils.fs.ls(RUTA_RAW)
    num_archivos = len(archivos_raw)
    print(f"  [RAW]    Archivos en volumen: {num_archivos}")
    for archivo in archivos_raw:
        print(f"           - {archivo.name} ({archivo.size:,} bytes)")
except Exception as e:
    print(f"  [RAW]    No se pudo acceder al volumen: {e}")

print()

# --- CAPA BRONCE: Contar registros en tablas Delta ---
tablas_bronce = [
    ("pubchem_bronce", "propiedades_compuestos"),
    ("pubchem_bronce", "bioactividad")
]

for esquema, tabla in tablas_bronce:
    try:
        conteo = spark.table(f"{esquema}.{tabla}").count()
        print(f"  [BRONCE] {esquema}.{tabla}: {conteo:,} registros")
    except Exception:
        print(f"  [BRONCE] {esquema}.{tabla}: No disponible")

print()

# --- CAPA PLATA: Contar registros en tabla consolidada ---
try:
    conteo_plata = spark.table("pubchem_plata.actividades_biologicas").count()
    print(f"  [PLATA]  pubchem_plata.actividades_biologicas: {conteo_plata:,} registros")
except Exception:
    print(f"  [PLATA]  pubchem_plata.actividades_biologicas: No disponible")

print()

# --- CAPA ORO: Contar registros en todas las tablas del modelo estrella ---
tablas_oro = [
    "dim_compuestos",
    "dim_ensayos",
    "dim_resultados",
    "fact_bioactividades"
]

for tabla in tablas_oro:
    try:
        conteo = spark.table(f"pubchem_oro.{tabla}").count()
        print(f"  [ORO]    pubchem_oro.{tabla}: {conteo:,} registros")
    except Exception:
        print(f"  [ORO]    pubchem_oro.{tabla}: No disponible")

print()
print("=" * 70)